In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df=pd.read_csv('../original_datasets/traffic_data.csv')

print(df.info())
print(df.describe())
print(df.head())

<class 'pandas.DataFrame'>
RangeIndex: 20328 entries, 0 to 20327
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   record_date        20328 non-null  str    
 1   hour_of_day        20328 non-null  str    
 2   city               20328 non-null  str    
 3   corridor           20328 non-null  str    
 4   congestion_index   20328 non-null  float64
 5   avg_speed_kmph     19967 non-null  str    
 6   incident_reported  20328 non-null  str    
 7   road_closure       18879 non-null  str    
dtypes: float64(1), str(7)
memory usage: 1.2 MB
None
       congestion_index
count      20328.000000
mean          10.268006
std           20.529642
min            0.030000
25%            0.363000
50%            0.528000
75%            0.782000
max           99.000000
  record_date hour_of_day     city        corridor  congestion_index  \
0  2024-01-01           7  Kolkata  Central Market             0.540   
1  2024-01

In [2]:
print(df.duplicated().sum())

600


In [3]:
df=df.drop_duplicates()

In [4]:
print(df.shape)
print(df.isna().sum())

(19728, 8)
record_date             0
hour_of_day             0
city                    0
corridor                0
congestion_index        0
avg_speed_kmph        347
incident_reported       0
road_closure         1409
dtype: int64


In [5]:
def clean_incident(val):
  if pd.isna(val):
    return '0'
  val_str = str(val).strip().upper()
  if val_str in ['TRUE', 'T', 'YES', 'Y', '1']:
    return '1'
  return '0'

df['incident_reported'] = df['incident_reported'].apply(clean_incident).astype(int)

# --- 2. Clean `road_closure` ---
def clean_closure(val):
  if pd.isna(val):
    return 'No'
  val_str = str(val).strip().upper()
  if val_str in ['YES', 'Y', 'TRUE', '1']:
    return 'Yes'
  return 'No'

df['road_closure'] = df['road_closure'].apply(clean_closure)

# --- 3. Clean `avg_speed_kmph` (Convert to numeric, handle 'null' strings) ---
df['avg_speed_kmph'] = pd.to_numeric(df['avg_speed_kmph'], errors='coerce')
# Fill missing speeds with median speed
df['avg_speed_kmph'] = df['avg_speed_kmph'].fillna(df['avg_speed_kmph'].median())

# --- 4. Clean `hour_of_day` (Remove ':00' if present and convert to int) ---
df['hour_of_day'] = (
    df['hour_of_day'].astype(str).str.replace(':00', '', regex=False)
)
df['hour_of_day'] = pd.to_numeric(df['hour_of_day'], errors='coerce').astype(int)

# --- 5. Cap Outliers in `congestion_index` using IQR ---
Q1 = df['congestion_index'].quantile(0.25)
Q3 = df['congestion_index'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - (1.5 * IQR)
upper_bound = Q3 + (1.5 * IQR)

df['congestion_index'] = np.clip(df['congestion_index'], lower_bound, upper_bound)
print(f'Capped `congestion_index` using upper bound: {upper_bound:.4f}')

# --- 6. Sort Chronologically ---
df['record_date'] = pd.to_datetime(df['record_date'])
df = df.sort_values(by=['city', 'record_date', 'hour_of_day']).reset_index(drop=True)

Capped `congestion_index` using upper bound: 1.4105


In [6]:
print(df.isna().sum())

record_date          0
hour_of_day          0
city                 0
corridor             0
congestion_index     0
avg_speed_kmph       0
incident_reported    0
road_closure         0
dtype: int64


In [7]:
df.to_csv('cleaned_traffic_data.csv',index=False)

In [8]:
num_cols = ['congestion_index', 'avg_speed_kmph']
eda_stats = {}

for col in num_cols:
    if col in df.columns:
        s = df[col].dropna()
        Q1 = s.quantile(0.25)
        Q3 = s.quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        outliers = s[(s < lower_bound) | (s > upper_bound)]
        
        eda_stats[col] = {
            'count': len(s),
            'mean': s.mean(),
            'std': s.std(),
            'median': s.median(),
            'min': s.min(),
            'max': s.max(),
            'q25': Q1,
            'q75': Q3,
            'iqr': IQR,
            'outlier_count': len(outliers),
            'lower_bound': lower_bound,
            'upper_bound': upper_bound
        }

for col, stats in eda_stats.items():
    print(f"--- {col} ---")
    for k, v in stats.items():
        print(f"  {k}: {v}")

--- congestion_index ---
  count: 19728
  mean: 0.6653737581103001
  std: 0.4274385542250339
  median: 0.528
  min: 0.03
  max: 1.4105
  q25: 0.363
  q75: 0.782
  iqr: 0.41900000000000004
  outlier_count: 0
  lower_bound: -0.26550000000000007
  upper_bound: 1.4105
--- avg_speed_kmph ---
  count: 19728
  mean: 32.7102798053528
  std: 11.699844511727264
  median: 32.7
  min: 4.0
  max: 68.5
  q25: 24.5
  q75: 40.9
  iqr: 16.4
  outlier_count: 4
  lower_bound: -0.09999999999999787
  upper_bound: 65.5


In [9]:
print(df.info())
print("\n--- Categorical Value Counts ---")
print("Cities:", df['city'].value_counts().to_dict())
print("Incidents Reported:", df['incident_reported'].value_counts().to_dict())
print("Road Closures:", df['road_closure'].value_counts().to_dict())
print("\n--- Correlation Matrix ---")
numeric_df = df.select_dtypes(include=[np.number])
print(numeric_df.corr())

<class 'pandas.DataFrame'>
RangeIndex: 19728 entries, 0 to 19727
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   record_date        19728 non-null  datetime64[us]
 1   hour_of_day        19728 non-null  int64         
 2   city               19728 non-null  str           
 3   corridor           19728 non-null  str           
 4   congestion_index   19728 non-null  float64       
 5   avg_speed_kmph     19728 non-null  float64       
 6   incident_reported  19728 non-null  int64         
 7   road_closure       19728 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(2), str(3)
memory usage: 1.2 MB
None

--- Categorical Value Counts ---
Cities: {'Ahmedabad': 1056, 'Kochi': 1056, 'Coimbatore': 1044, 'Bengaluru': 1020, 'Delhi': 1020, 'Hyderabad': 1020, 'Gurugram': 1014, 'Jaipur': 984, 'Nagpur': 984, 'Madurai': 978, 'Kolkata': 972, 'Mumbai': 972, 'Pune': 972, 'Chandigarh':

In [10]:
df['avg_speed_kmph'] = pd.to_numeric(df['avg_speed_kmph'], errors='coerce')
df['congestion_index'] = pd.to_numeric(df['congestion_index'], errors='coerce')

city_summary = df.groupby('city')[['congestion_index', 'avg_speed_kmph']].mean().reset_index()
print(city_summary.sort_values(by='congestion_index', ascending=False).head(10))

             city  congestion_index  avg_speed_kmph
8        Guwahati          0.706425       32.491667
19  Visakhapatnam          0.681761       32.768333
1       Bengaluru          0.680536       32.574314
4         Chennai          0.674941       32.173270
5      Coimbatore          0.673681       32.706801
17         Nagpur          0.670765       32.493089
6           Delhi          0.668965       33.000686
9       Hyderabad          0.668750       32.305294
3      Chandigarh          0.664599       32.325729
16         Mysuru          0.664059       32.970545
